In [ ]:
"""
    A practical example for fine-tuning an open-weight LLM using LoRA with Hugging Face Transformers and PEFT (Parameter-Efficient Fine-Tuning)

    We will use:
        - LLaMA-style workflow
        - LoRA (efficient fine-tuning)
        - Small instruction dataset
        - Single GPU setup
"""

In [ ]:
# Dependencies
# !pip install transformers datasets peft accelerate bitsandbytes trl ipywidgets hf_transfer
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"              # disable tokenizer multi-threading (parallelism)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"               # force fast Rust-based downloader
os.environ["HF_HUB_ENABLE_DAG_PROCESSOR"] = "1"

import torch
from huggingface_hub.utils import enable_progress_bars

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig, PeftModel
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer                       # supervised fine-tuning trainer

# Enable HF progress bars
enable_progress_bars()

# Quick sanity check on GPU
print(f">>> Using PyTorch version: {torch.__version__}")
print(f">>> CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f">>> GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Model: 8-billion-parameter Llama 3 model pre-quantized using BitsAndBytes into the 4-bit nf4 format
model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"         

# Tokenizer Setup
print(">>> Downloading/Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Fix Llama 3 padding trap: Use the dedicated fine-tuning pad token
if "<|finetune_right_pad_id|>" in tokenizer.get_vocab():
    tokenizer.pad_token = "<|finetune_right_pad_id|>"
else:
    # Fallback if the specific token isn't found
    tokenizer.add_special_tokens({"pad_token": "<|pad|>"})

# Crucial for SFTTrainer training alignment
tokenizer.padding_side = "right" 
print(f">>> Tokenizer loaded! Pad Token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

# Quantization & Model Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True # Saves an extra bit of VRAM per parameter
)

print(">>> Downloading/Loading Model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto", 
    torch_dtype=torch.bfloat16
)

model.resize_token_embeddings(len(tokenizer))

# Prepare layers for LoRA training
model = prepare_model_for_kbit_training(model)
print(">>> Model loaded and prepared for LoRA successfully!")

In [ ]:
# LoRA (Low-Rank Adaptation) Configuration
lora_config = LoraConfig(
    r=16,                                           # the size of the low-rank matrices injected into the model
    lora_alpha=32,                                  # scaling factor for the LoRA weights, normally set to double the value of r
    target_modules=[
        "q_proj", 
        "k_proj", 
        "v_proj", 
        "o_proj", 
        "gate_proj", 
        "up_proj", 
        "down_proj"
    ],                                              # for llama-3 model, target all linear modules
    lora_dropout=0.00,                              
    bias="none",                                    # bias parameters will not be trained
    task_type="CAUSAL_LM"                           # training a Causal Language Model
)

# Wrap the quantized base model with the LoRA adapters
model = get_peft_model(model, lora_config)

# Helper: Enables the gradients for the input embeddings, necessary for stable mixed-precision training
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
else:
    model.get_input_embeddings().register_forward_hook(
        lambda module, input, output: output.requires_grad_(True)
    )

# Sanity Checks
print(">>> Model training device:", next(model.parameters()).device)
print(">>> Trainable parameters breakdown:")
model.print_trainable_parameters()

In [ ]:
# Load the raw dataset
dataset = load_dataset("json", data_files="train.json")

# Define a clean, row-by-row formatting function
def formatting_prompts_func(examples):
    output_texts = []
    # Loop through the batch elements provided by SFTTrainer natively
    for i in range(len(examples['input'])):
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": examples['input'][i]},
            {"role": "assistant", "content": examples['output'][i]}
        ]
        # Format using Llama-3 templates (tokenize=False, add_generation_prompt=False for training)
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        output_texts.append(text)
        
    return output_texts

print(">>> Dataset wrapper function configured successfully!")

In [ ]:
# Trainer Configuration
training_args = SFTConfig(
    output_dir="./results",                                 # training checkpoint saving directory
    per_device_train_batch_size=2,                          # number of training examples processed simultaneously on a single GPU
    gradient_accumulation_steps=3,                          # accumulate the gradients for 3 steps before performing a weight update -> effective batch size = 6
    learning_rate=2e-4,                                     # peak learning rate
    lr_scheduler_type="cosine",                             # smoothly decays the LR using a cosine curve
    warmup_ratio=0.03,                                      # warm up for the first 3% of total steps
    num_train_epochs=3,                                     # number of epochs
    logging_steps=1,                                        # print training metrics every step
    save_strategy="epoch",                                  # save a checkpoint automatically at the end of every epoch
    save_total_limit=2,                                     # automatically keeps only the 2 most recent checkpoints
    fp16=False,
    bf16=True,
    max_length=512,
    packing=True,                                           # pack short sequences together to drastically boost training speed
    gradient_checkpointing=True,                            # gradient checkpointing saves massive VRAM by recalculating activations
    gradient_checkpointing_kwargs={"use_reentrant": False}  # prevent PyTorch warning
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    formatting_func=formatting_prompts_func, 
    args=training_args,
    processing_class=tokenizer
)

print(">>> Trainer initialized and optimized for speed!")

In [ ]:
# Execution & Training Loop
print(">>> Starting training...")

# Launch the training process
training_results = trainer.train()

print(">>> Training complete!")
print(f">>> Final Train Loss: {training_results.training_loss:.4f}")

# Save the trained LoRA adapters and tokenizer configurations 
final_model_path = "./final_lora_adapter"
trainer.model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f">>> LoRA adapters successfully saved to: {final_model_path}")

In [ ]:
# Base Model & Tokenizer Reloading - CRITICAL: Use the unquantized base model repo for a 16-bit merge
base_model_name = "meta-llama/Meta-Llama-3-8B-Instruct" 
adapter_dir = "./final_lora_adapter" 

print(">>> Reloading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

print(">>> Reloading Base Model in full 16-bit precision...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True              # prevent Jupyter kernel crash from RAM spikes
)

# Adapter Loading & Merging
print(">>> Attaching LoRA Adapter layers...")
model = PeftModel.from_pretrained(base_model, adapter_dir)

print(">>> Merging weights permanently (this may take a minute)...")
merged_model = model.merge_and_unload(safe_merge=True)      # safe_merge=True handles edge-case mathematical alignment safely

# Final Export
output_dir = "./final_merged_model_bf16"
print(f">>>> Saving complete standalone model to: {output_dir}")

merged_model.save_pretrained(output_dir, max_shard_size="5GB")
tokenizer.save_pretrained(output_dir)

print(">>> Process complete! Your model is fully standalone and ready for high-speed inference.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Path to your fully combined 16-bit standalone model
model_path = "./final_merged_model_bf16"

# Load Tokenizer (Correct Padding for Inference)
print(">>> Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Retain the same padding safety from training
if "<|finetune_right_pad_id|>" in tokenizer.get_vocab():
    tokenizer.pad_token = "<|finetune_right_pad_id|>"
else:
    tokenizer.pad_token = tokenizer.eos_token

# CRITICAL FOR INFERENCE: Padding side must be "left" for text generation loops
tokenizer.padding_side = "left"

# Load the Standalone Merged Model
print(">>> Loading Merged Model into VRAM...")

try:
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",    # Ultra-fast execution
        device_map="auto",                          # Dynamic allocation safely mapped
        low_cpu_mem_usage=True
    )
    print(">>> Successfully loaded with FlashAttention-2!")
except Exception as e:
    print(f">>> FlashAttention-2 loading failed or unsupported. falling back to SDPA... \nError: {e}")
    # Fallback to PyTorch Native SDPA (highly optimized, works everywhere)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        attn_implementation="sdpa",
        device_map="auto",
        low_cpu_mem_usage=True
    )

print(">>> Model Target Device:", model.device)

In [ ]:
# Create a test message using Llama-3 instruction structure
question = "How to optimize a Live Image?"

eval_messages = [
    {
        "role": "system", 
        "content": (
            "You are a strict, factual assistant. Answer the user's question using ONLY "
            "the specific knowledge you were fine-tuned on. If the answer to the question "
            "cannot be found in your fine-tuning data, you must reply exactly with: "
            "'I do not have sufficient information to answer this question.' Do not invent "
            "or assume anything."
        )
    },
    {
        "role": "user", 
        "content": question
    }
]

# Apply template with inference format (add_generation_prompt=True tells the model it is its turn to speak)
prompt = tokenizer.apply_chat_template(eval_messages, tokenize=False, add_generation_prompt=True)

# Tokenize inputs and cast them onto the GPU
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
input_ids = inputs["input_ids"] 
input_length = input_ids.shape[1]

# Generate response text safely
print("\n>>> Generating...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,         # Force strict Greedy Search execution                 
        eos_token_id=tokenizer.eos_token_id
    )

# Decode output and isolate the assistant reply
answer = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
print(">>> QUESTION: ", question)
print("\n>>> ANSWER FROM FINE-TUNED LLM: ", answer.split("assistant")[-1].strip())